# 전체 PDF 청킹과 Supabase DB 적재

**전략:** 작은 child 청크는 `chunks`, 부모 섹션 전체 본문은 `sections`에 저장합니다. 검색 시 문서 ID와 소제목 ID로 부모를 가져옵니다.

기준: [RAG 검색 전략: 작은 청크와 부모 섹션](https://app.notion.com/p/RAG-3f0aa196a18280828414ed3f5c2418a5).

이 노트북은 1번의 테이블을 사용합니다. 기존 소제목 청킹 학습 노트북은 한 페이지 예제로 유지합니다. 전체 PDF의 공식 북마크를 부모 경계로 쓰므로, 기존 학습 노트북에서 직접 지정한 작은 소제목과 부모 경계가 다를 수 있습니다. 예를 들어 ‘비상시 충전 커넥터 분리하기’는 ‘전기차 충전 정보 확인하기’ 부모 안에 포함됩니다.

**범위:** 전체 페이지 읽기, 부모·자식 청킹, 삽입 이미지의 표시 영역 저장, Storage 업로드, DB 등록, 검증. 임베딩 생성과 LLM 답변은 다음 단계입니다. 청크의 임베딩 3개 항목은 NULL로 시작합니다.

**재실행:** 같은 파일·같은 청킹 설정의 결과는 중복 없이 다시 등록합니다. 이미 저장된 문서의 내용이나 설정이 다르면 중단합니다. 다른 팀의 테이블을 수정하지 않습니다. PDF 파일을 수정하지 않습니다.

In [ ]:
# 필요한 도구를 가져옵니다. PyMuPDF는 페이지의 글자 위치와 북마크를 읽기 위해 추가로 사용합니다.
from pathlib import Path
from collections import Counter
from bisect import bisect_right
import os, sys, re, json, hashlib, time
import pymupdf
import pandas as pd
from PIL import Image as PILImage
from dotenv import load_dotenv
from psycopg.conninfo import conninfo_to_dict
from psycopg import sql
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, Image

# 팀 공통 연결 도구를 찾을 수 있도록 src 경로를 알려줍니다.
project_root = Path('/Users/anna/mle-02-p1-team1')
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))
from car_search_rag.common.database_manager import DatabaseManager
from car_search_rag.common.storage_manager import StorageManager
load_dotenv(project_root / '.env', override=False)

# 접속 정보는 출력하지 않고, 팀 프로젝트인지 확인합니다.
connection_info = conninfo_to_dict(os.environ['DB_URL'])
project_ref = 'hbuvewommtcmbbynsvht'
assert project_ref in connection_info.get('host', '') or project_ref in connection_info.get('user', ''), 'DB 접속 대상을 확인하세요.'
assert project_ref in os.environ.get('SUPABASE_URL', ''), 'Storage 접속 대상을 확인하세요.'
os.environ.setdefault('PGCONNECT_TIMEOUT', '10')
db = DatabaseManager()
storage = StorageManager('images')
print('DB와 Storage 연결 도구 준비 완료')

## 1. 처리 설정과 안정적인 ID

파일의 바이트로 SHA-256 해시를 계산합니다. 같은 파일이면 같은 문서 ID를 사용합니다. 청킹 설정이 바뀌었을 때 기존 결과와 조용히 섞이지 않도록 처리 버전도 기록합니다. 수업에서 사용한 분할 도구를 쓰며 시작값은 본문 400자, 최대 목표 겹침 80자입니다. 실제 겹침은 문단 경계에 따라 달라집니다.

In [ ]:
pdf_path = Path('/Users/anna/Downloads/NE1_2027_ko_KR.pdf')
file_sha256 = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
source_id = pdf_path.stem
# 개정 PDF와 구분하면서 같은 파일 재실행 시 ID를 유지합니다.
document_id = f'{source_id}_{file_sha256[:12]}'
vehicle_id = 'ioniq5'
chunk_size, chunk_overlap = 400, 80
pipeline_version = 'bookmark_parent_v1_child400_overlap80'
output_dir = project_root / 'src/car_search_rag/anna_rag/data/full_manual' / document_id
output_dir.mkdir(parents=True, exist_ok=True)
image_dir = output_dir / 'images'
image_dir.mkdir(exist_ok=True)
pdf = pymupdf.open(pdf_path)
print('전체 페이지:', len(pdf), '/ 북마크:', len(pdf.get_toc()))
print('문서 ID:', document_id)
print('처리 결과 폴더:', output_dir)

## 2. 모든 페이지에서 위치가 있는 글자 읽기

PyMuPDF로 읽은 글자를 좌표 순서로 정렬합니다. 같은 줄의 번호·글머리 기호를 본문 앞에 붙이고, 페이지 하단의 반복 쪽수·머리말은 제외합니다. 그림 아래의 2pt 식별 문자열은 본문 검색에서 제외하고 이미지 연결의 보조 자료로 남깁니다.

경고 아이콘 옆 전용 글꼴이 깨져 추출되는 경우에만 라벨을 복원합니다. 일반 본문의 글자는 임의로 치환하지 않습니다. 표는 좌표상 같은 줄의 글자를 탭으로 이어 보관하며, 복잡한 표 구조의 완벽한 복원이나 OCR은 수행하지 않습니다.

In [ ]:
def normalized_title(text):
    # 줄바꿈과 공백만 제거하여 제목을 비교합니다.
    return re.sub(r'\s+', '', text)


def clean_span(span):
    text = span['text']
    # 이 PDF의 특수 글꼴 라벨만 복원합니다. 짧은 글자를 전체 본문에서 replace하지 않습니다.
    if 'HyundaiSansTextKRMedium-' in span['font']:
        compact = text.strip()
        if 'Ҋ' in compact:
            return '[경고]'
        if '઱' in compact or '\u0ab1' in compact:
            return '[주의]'
        if 'ঌ' in compact or 'ف' in compact or 'ӝ' in compact:
            return '[알아두기]'
    return text


page_records = []
for page_index, page in enumerate(pdf):
    # 이미지 바이트는 여기서 읽지 않아 메모리 사용을 줄입니다.
    data = page.get_text('dict', flags=pymupdf.TEXTFLAGS_DICT & ~pymupdf.TEXT_PRESERVE_IMAGES)
    raw_lines, markers, footer = [], [], []
    for block in data['blocks']:
        for line in block.get('lines', []):
            spans = line['spans']
            text = ''.join(clean_span(span) for span in spans).strip()
            if not text:
                continue
            size = max(span['size'] for span in spans)
            x0, y0, x1, y1 = line['bbox']
            if size <= 3:
                markers.append({'text': text, 'bbox': list(line['bbox'])})
                continue
            # 본문 아래 9.5pt 반복 쪽수만 제거합니다.
            if y0 >= 550 and 9 <= size <= 10:
                footer.append(text)
                continue
            raw_lines.append({'text': text, 'x': x0, 'y': y0, 'bottom': y1, 'size': size})
    raw_lines.sort(key=lambda row: (row['y'], row['x']))
    # 기준 y가 2pt 이내이면 같은 줄로 묶어 글머리 기호와 내용을 합칩니다.
    groups = []
    for row in raw_lines:
        if not groups or abs(row['y'] - groups[-1][0]['y']) > 2:
            groups.append([row])
        else:
            groups[-1].append(row)
    lines = []
    for group in groups:
        group.sort(key=lambda row: row['x'])
        lines.append({'text': '\t'.join(row['text'] for row in group),
                      'y': min(row['y'] for row in group),
                      'bottom': max(row['bottom'] for row in group)})
    printed_page = next((text for text in footer if text.isdigit()), None)
    page_records.append({'pdf_page': page_index + 1, 'lines': lines, 'raw_lines': raw_lines,
                         'markers': markers, 'printed_page': printed_page})
print('읽은 페이지:', len(page_records))
print('18페이지 첫 5줄:', [row['text'] for row in page_records[17]['lines'][:5]])

## 3. PDF 공식 북마크로 부모 섹션 나누기

북마크는 PDF 작성자가 넣어둔 목차 정보입니다. 제목과 본문 위치를 대조한 뒤 경계를 정합니다. 다음 북마크 전까지의 본문을 한 부모에 넣고 10pt 하위 소제목·경고·주의는 그 안에 유지합니다. 페이지가 넘어가도 같은 부모를 유지합니다.

이 PDF의 인쇄 목차 페이지는 5~8페이지입니다. 본문과 삽입 이미지가 없는 페이지도 검색 대상에서 제외합니다. 페이지 내용은 별도로 보관하되 검색 청크에서는 제외합니다. 제외 페이지 목록을 다른 PDF에 그대로 재사용하면 안 됩니다. 모든 페이지를 읽었다는 것과 모든 페이지를 검색 대상에 넣었다는 것은 구분합니다.
531페이지는 본문 텍스트가 없는 독립된 제품 표시사항 이미지입니다. 원본 화면에서 확인한 제목을 수동 전사하여 별도 부모로 등록합니다. 전체 목차 551개에 이 예외 1개를 더하며, 상세 글자는 OCR 처리하지 않습니다.


In [ ]:
# 이 PDF에서 직접 확인한 인쇄 목차 페이지입니다.
toc_pages = set(range(5, 9))
blank_pages = {row['pdf_page'] for row in page_records
               if not row['lines'] and not pdf[row['pdf_page'] - 1].get_image_info()}
excluded_search_pages = toc_pages | blank_pages
heading_records, title_stack = [], []
for toc_order, (level, title, page_no, destination) in enumerate(pdf.get_toc(False), start=1):
    record = page_records[page_no - 1]
    candidates = sorted([row for row in record['raw_lines'] if row['size'] >= 10], key=lambda row: (row['y'], row['x']))
    matches = []
    # 두세 줄로 나뉜 긴 제목도 같은 제목으로 비교합니다.
    for index, row in enumerate(candidates):
        joined = ''
        for following in candidates[index:index + 3]:
            joined += following['text']
            if normalized_title(joined) == normalized_title(title):
                matches.append(row['y'])
    if not matches:
        raise ValueError(f'PDF {page_no}페이지 목차 제목 확인 필요: {title}')
    target_y = destination['to'].y
    actual_y = min(matches, key=lambda y: abs(y - target_y))
    if abs(actual_y - target_y) > 5:
        raise ValueError(f'목차와 본문 위치 차이 확인 필요: {title}')
    title_stack = title_stack[:level - 1] + [title]
    heading_records.append({'page': page_no, 'y': actual_y, 'title': title,
                            'path': title_stack.copy(), 'order': toc_order})
# 531페이지는 원본 화면에서 확인한 독립된 이미지 문서입니다.
# 이미지 안의 제목을 수동 전사합니다. OCR이나 LLM 자동 캡션을 실행하는 코드는 아닙니다.
reviewed_image_only_pages = {531: '안전확인대상생활화학제품 표시사항'}
for page_no, title in reviewed_image_only_pages.items():
    assert not page_records[page_no - 1]['lines'], '수동 예외 페이지의 구조가 바뀌었습니다.'
    heading_records.append({'page': page_no, 'y': 0.0, 'title': title,
                            'path': ['차량의 유지보수', title], 'order': len(heading_records) + 1})
    page_records[page_no - 1]['lines'] = [{'text': title, 'y': 0.0, 'bottom': 0.0}]
heading_records.sort(key=lambda row: (row['page'], row['y'], row['order']))
heading_positions = [(row['page'], row['y'] - 0.1) for row in heading_records]
assert len(set(heading_positions)) == len(heading_positions), '같은 위치의 목차는 수동 확인이 필요합니다.'

# 각 부모의 문장 목록을 준비합니다.
section_lines = [[] for _ in heading_records]
excluded_pages = []
for record in page_records:
    if record['pdf_page'] in excluded_search_pages:
        excluded_pages.append({'pdf_page': record['pdf_page'], 'reason': '인쇄 목차' if record['pdf_page'] in toc_pages else '본문 없는 빈 페이지',
                               'text': '\n'.join(row['text'] for row in record['lines'])})
        continue
    for line in record['lines']:
        parent_index = bisect_right(heading_positions, (record['pdf_page'], line['y'] + 0.1)) - 1
        if parent_index < 0:
            raise ValueError('첫 목차 이전 본문을 확인하세요.')
        section_lines[parent_index].append({**line, 'page': record['pdf_page']})

sections, section_spans = [], {}
for index, (heading, lines) in enumerate(zip(heading_records, section_lines)):
    if not lines:
        raise ValueError(f"내용이 없는 부모: {heading['title']}")
    section_id = f's{index + 1:04d}_p{heading["page"]:03d}'
    full_text, spans = '', []
    previous = None
    for line in lines:
        # 시각적으로 떨어진 문단은 빈 줄을 두어 splitter가 문단 경계를 우선 사용하게 합니다.
        gap = '\n\n' if previous and (previous['page'] != line['page'] or line['y'] - previous['bottom'] > 4) else '\n'
        if full_text:
            full_text += gap
        start = len(full_text)
        full_text += line['text']
        spans.append({'start': start, 'end': len(full_text), 'page': line['page'], 'y': line['y']})
        previous = line
    pages = sorted({line['page'] for line in lines})
    printed = [page_records[page - 1]['printed_page'] for page in pages]
    sections.append({'document_id': document_id, 'section_id': section_id, 'title': heading['title'],
                     'section_path': heading['path'], 'full_text': full_text, 'pdf_pages': pages,
                     'printed_pages': [page for page in printed if page is not None] or None,
                     'section_order': index + 1})
    section_spans[section_id] = spans
# 제외 페이지도 로컬에 보관해 전체 페이지 처리 내역을 남깁니다.
(output_dir / 'excluded_pages.json').write_text(json.dumps(excluded_pages, ensure_ascii=False, indent=2))
print('부모 섹션:', len(sections), '/ 검색 제외 목차·빈 페이지:', sorted(excluded_search_pages))
display(pd.DataFrame(sections)[['section_id', 'title', 'pdf_pages']].head(12))

## 4. 부모 안에서 작은 child 청크 만들기

`RecursiveCharacterTextSplitter`는 빈 줄, 줄바꿈, 문장 끝, 공백 순으로 나눌 위치를 찾습니다. 한 부모에서 만든 청크가 다른 부모와 섞이지 않도록 부모마다 따로 실행합니다. 청크에는 목차 경로를 붙여 대상과 맥락을 보완합니다.

400자는 본문 기준입니다. 제목 경로가 추가된 `page_content`는 더 길 수 있습니다. 출처 페이지는 부모의 전체 페이지를 무조건 복사하지 않고, 실제 청크의 문자 구간과 페이지 구간을 대조해 기록합니다.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap,
    separators=['\n\n', '\n', '. ', ' ', ''],
    add_start_index=True,
)
text_chunks, chunk_audit = [], []
for section in sections:
    parent_doc = Document(page_content=section['full_text'], metadata={'section_id': section['section_id']})
    children = splitter.split_documents([parent_doc])
    for order, child in enumerate(children, start=1):
        start = child.metadata['start_index']
        end = start + len(child.page_content)
        assert start >= 0 and section['full_text'][start:end] == child.page_content
        pages = sorted({span['page'] for span in section_spans[section['section_id']]
                        if span['start'] < end and span['end'] > start})
        chunk_id = f"{document_id}_{section['section_id']}_text_{order:03d}"
        text_chunks.append({'chunk_id': chunk_id, 'document_id': document_id,
                            'section_id': section['section_id'], 'chunk_type': 'text',
                            'chunk_order': order,
                            'page_content': '목차: ' + ' > '.join(section['section_path']) + '\n본문:\n' + child.page_content,
                            'pdf_pages': pages, 'image_id': None})
        chunk_audit.append({'chunk_id': chunk_id, 'section_id': section['section_id'],
                            'start': start, 'end': end, 'body_length': len(child.page_content), 'pdf_pages': pages})
print('텍스트 청크:', len(text_chunks))
display(pd.DataFrame(chunk_audit).head())

## 5. 삽입 이미지와 주변 글 연결

PDF에 삽입된 이미지의 화면 영역을 2배 해상도로 PNG 렌더링합니다. 해당 영역 안에 PDF가 덧그린 글자·선을 함께 보존하려는 목적입니다. 이미지 영역 바깥의 표식이나 별도의 벡터 그림 전체를 자동 수집하는 방식은 아닙니다.

캡션은 `소제목 + 주변의 실제 본문`을 사용하며 `caption_source='surrounding_text'`로 표시합니다. 그림을 AI가 해석한 설명이 아닙니다. 수동 캡션은 별도 딕셔너리에 명시합니다. 작은 아이콘도 임의로 버리지 않고 위치와 크기를 검토 목록에 남깁니다.

In [ ]:
# 이미 직접 확인했던 18페이지 그림의 캡션입니다. 키는 (PDF 페이지, 해당 페이지 이미지 순서)입니다.
manual_captions = {(18, 1): '화물칸의 충전 커넥터 비상 해제 케이블을 확대해 보여주는 그림',
                   (531, 1): '필터형 보존처리 제품의 안전확인대상생활화학제품 표시사항. 성분, 사용방법, 주의사항은 원본 그림에서 확인합니다.'}
images, section_images, image_chunks, image_audit = [], [], [], []
image_orders = Counter()
for page_index, page in enumerate(pdf):
    page_no = page_index + 1
    if page_no in excluded_search_pages:
        assert not page.get_image_info(), '검색 제외 페이지의 이미지를 별도로 확인하세요.'
        continue
    # 위에서 아래, 왼쪽에서 오른쪽으로 그림 순서를 고정합니다.
    image_infos = sorted(page.get_image_info(xrefs=True), key=lambda item: (item['bbox'][1], item['bbox'][0], item['number']))
    for image_no, info in enumerate(image_infos, start=1):
        box = pymupdf.Rect(info['bbox']) & page.rect
        if box.is_empty or box.is_infinite:
            raise ValueError(f'그림 영역 오류: {page_no}/{image_no}')
        parent_index = bisect_right(heading_positions, (page_no, box.y0)) - 1
        assert parent_index >= 0
        section = sections[parent_index]
        image_id = f'p{page_no:03d}_image_{image_no:02d}'
        file_name = f'{image_id}.png'
        local_path = image_dir / file_name
        clip = pymupdf.Rect(box.x0 - 2, box.y0 - 2, box.x1 + 2, box.y1 + 2) & page.rect
        page.get_pixmap(matrix=pymupdf.Matrix(2, 2), clip=clip, alpha=False).save(local_path)
        # 다른 소제목의 글이 캡션에 섞이지 않도록 같은 부모의 줄만 후보로 사용합니다.
        same_parent_lines = [row for row in section_lines[parent_index] if row['page'] == page_no]
        nearby = sorted(same_parent_lines, key=lambda row: min(abs(row['y'] - box.y1), abs(row['bottom'] - box.y0)))[:5]
        nearby.sort(key=lambda row: row['y'])
        context = '\n'.join(row['text'] for row in nearby)[:500]
        caption = manual_captions.get((page_no, image_no))
        caption_source = 'manual' if caption else 'surrounding_text'
        caption = caption or f"{section['title']}의 그림. 주변 본문(자동 연결, 이미지 해석 아님):\n{context}"
        storage_path = f'cars/hyundai/ioniq5/{document_id}/{file_name}'
        images.append({'document_id': document_id, 'image_id': image_id, 'caption': caption,
                       'caption_source': caption_source, 'pdf_page': page_no,
                       'storage_bucket': 'images', 'storage_path': storage_path})
        section_images.append({'document_id': document_id, 'section_id': section['section_id'], 'image_id': image_id})
        image_orders[section['section_id']] += 1
        image_chunks.append({'chunk_id': f"{document_id}_{section['section_id']}_{image_id}",
                             'document_id': document_id, 'section_id': section['section_id'],
                             'chunk_type': 'image', 'chunk_order': image_orders[section['section_id']],
                             'page_content': '목차: ' + ' > '.join(section['section_path']) + '\n이미지 설명: ' + caption,
                             'pdf_pages': [page_no], 'image_id': image_id})
        image_audit.append({'image_id': image_id, 'pdf_page': page_no, 'section_id': section['section_id'],
                            'bbox': list(box), 'width': box.width, 'height': box.height,
                            'local_path': str(local_path), 'sha256': hashlib.sha256(local_path.read_bytes()).hexdigest()})
# 본문 없이 그림만 있는 페이지도 부모의 출처 목록에 포함합니다.
for section in sections:
    image_pages = {row['pdf_pages'][0] for row in image_chunks if row['section_id'] == section['section_id']}
    section['pdf_pages'] = sorted(set(section['pdf_pages']) | image_pages)
    section['printed_pages'] = [page_records[page - 1]['printed_page'] for page in section['pdf_pages']
                                if page_records[page - 1]['printed_page'] is not None] or None
all_chunks = text_chunks + image_chunks
print('그림:', len(images), '/ 전체 검색 청크:', len(all_chunks))
display(Image(filename=str(image_dir / 'p018_image_01.png')))

## 6. 업로드 전에 전체 연결과 누락 검사

읽은 본문의 글자들이 child 청크들에 모두 포함되는지, 실제 출처 페이지가 맞는지, 이미지 파일이 정상인지 확인합니다. 이 검사는 자동 추출의 일관성 검사이며, 565페이지를 사람이 모두 검수했다는 뜻은 아닙니다.

In [ ]:
# 부모 본문의 공백이 아닌 모든 글자가 적어도 한 child 청크에 포함되는지 확인합니다.
for section in sections:
    covered = bytearray(len(section['full_text']))
    for row in chunk_audit:
        if row['section_id'] == section['section_id']:
            covered[row['start']:row['end']] = b'1' * (row['end'] - row['start'])
    missing = [index for index, char in enumerate(section['full_text']) if not char.isspace() and covered[index] == 0]
    assert not missing, f"청크 본문 누락: {section['section_id']}"
assert len({row['chunk_id'] for row in all_chunks}) == len(all_chunks)
assert len({row['image_id'] for row in images}) == len(images)
assert set(range(1, len(pdf) + 1)) == {page for row in sections for page in row['pdf_pages']} | excluded_search_pages
assert all(row['pdf_pages'] and row['page_content'].strip() for row in all_chunks)
for row in image_audit:
    with PILImage.open(row['local_path']) as image_file:
        image_file.verify()
# 기존 학습 노트북에서 학습한 비상 충전 해제 내용과 경고·주의가 부모에 함께 있는지 확인합니다.
emergency_parent = next(row for row in sections if '비상시 충전 커넥터 분리하기' in row['full_text'])
assert '[경고]' in emergency_parent['full_text'] and '[주의]' in emergency_parent['full_text']
print('비상 충전 해제 부모:', emergency_parent['title'], emergency_parent['pdf_pages'])
manual = {'document_id': document_id, 'vehicle_id': vehicle_id, 'model_year': 2027,
          'document_version': pipeline_version, 'source_file': pdf_path.name, 'file_sha256': file_sha256}
tables = {'manuals': [manual], 'sections': sections, 'images': images,
          'section_images': section_images, 'chunks': all_chunks}
# API 키나 접속 정보 없이 추출 결과만 로컬 JSON에 저장합니다.
(output_dir / 'records.json').write_text(json.dumps(tables, ensure_ascii=False, indent=2))
(output_dir / 'chunk_audit.json').write_text(json.dumps(chunk_audit, ensure_ascii=False, indent=2))
(output_dir / 'image_audit.json').write_text(json.dumps(image_audit, ensure_ascii=False, indent=2))
summary = {'pages_read': len(pdf), 'pages_excluded_from_search': sorted(excluded_search_pages),
           'parent_sections': len(sections), 'text_chunks': len(text_chunks), 'image_chunks': len(image_chunks),
           'images': len(images), 'caption_sources': dict(Counter(row['caption_source'] for row in images)),
           'largest_parent_characters': max(len(row['full_text']) for row in sections),
           'pipeline_version': pipeline_version, 'embeddings_generated': False,
           'manual_image_only_section_pages': list(reviewed_image_only_pages),
           'printed_toc_pages': sorted(toc_pages), 'blank_pages': sorted(blank_pages)}
(output_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.DataFrame([{'테이블': name, '등록 예정': len(rows)} for name, rows in tables.items()]))
print('로컬 누락·연결·파일 검사 통과')

## 7. 기존 DB와 충돌 확인

다른 실행에서 이미 저장한 데이터가 있으면 같은 값인지 비교합니다. 같은 문서가 다른 ID로 등록되어 있거나, 기존 값이 현재 추출 결과와 다르면 자동으로 지우거나 덮어쓰지 않고 중단합니다. 기존 임베딩은 이 비교에서 제외하고 보존합니다.

In [ ]:
keys = {'manuals': ['document_id'], 'sections': ['document_id', 'section_id'],
        'images': ['document_id', 'image_id'], 'section_images': ['document_id', 'section_id', 'image_id'],
        'chunks': ['chunk_id']}

def check_existing(conn):
    # 같은 PDF 내용이 다른 문서 ID로 이미 등록되어 있는지 확인합니다.
    duplicate = conn.execute('SELECT document_id FROM anna_rag.manuals WHERE file_sha256=%s', (file_sha256,)).fetchone()
    if duplicate and duplicate['document_id'] != document_id:
        raise ValueError('같은 PDF가 다른 ID로 등록되어 있습니다. 기존 ID와 먼저 통합하세요.')
    for name, rows in tables.items():
        columns = list(rows[0])
        query = sql.SQL('SELECT {} FROM anna_rag.{} WHERE document_id=%s').format(
            sql.SQL(',').join(map(sql.Identifier, columns)), sql.Identifier(name))
        existing = conn.execute(query, (document_id,)).fetchall()
        expected = {tuple(row[key] for key in keys[name]): row for row in rows}
        for row in existing:
            identity = tuple(row[key] for key in keys[name])
            if identity not in expected or row != expected[identity]:
                raise ValueError(f'{name}에 현재 결과와 다른 데이터가 있습니다. 자동 덮어쓰기를 중단합니다.')

with db.connect(camel_case_keys=False) as conn:
    conn.execute('SET TRANSACTION READ ONLY')
    check_existing(conn)
print('기존 데이터 충돌 없음')

## 8. 실제 Supabase Storage 업로드

**이 셀은 외부 Storage에 PNG 파일을 업로드합니다.** 문서 해시가 포함된 하위 경로를 사용합니다. 로컬 업로드 기록이 있으면 서버 파일의 실제 바이트 해시를 확인하고 일치하는 파일은 건너뜁니다. 네트워크 오류로 중단되어도 다시 실행할 수 있습니다.

Storage와 DB는 하나의 트랜잭션이 아닙니다. DB 적재가 실패하더라도 성공한 파일 업로드는 남습니다. 업로드 기록을 보관해 재시도하고, 다른 파일은 삭제하지 않습니다.

In [ ]:
upload_log_path = output_dir / 'upload_log.json'
upload_log = json.loads(upload_log_path.read_text()) if upload_log_path.exists() else {}
image_lookup = {row['image_id']: row for row in images}
uploaded, reused = 0, 0
for index, row in enumerate(image_audit, start=1):
    info = image_lookup[row['image_id']]
    object_path = info['storage_path']
    verified = False
    if upload_log.get(object_path) == row['sha256']:
        try:
            remote_bytes = storage.supabase.storage.from_('images').download(object_path)
            verified = hashlib.sha256(remote_bytes).hexdigest() == row['sha256']
        except Exception:
            verified = False
    if verified:
        reused += 1
    else:
        # 팀 공통 업로드 도구를 사용합니다. 문서 하위 폴더를 포함한 이름을 전달합니다.
        for attempt in range(3):
            try:
                storage.upload_car_image(row['local_path'], 'hyundai', 'ioniq5', f"{document_id}/{row['image_id']}.png")
                break
            except Exception:
                if attempt == 2:
                    raise RuntimeError(f'이미지 업로드 실패: {row["image_id"]}. 이 셀을 다시 실행하세요.') from None
                time.sleep(1 + attempt)
        uploaded += 1
    upload_log[object_path] = row['sha256']
    # 매번 원자적으로 로그를 저장해 중단 시 진행 내역을 보존합니다.
    temporary = upload_log_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(upload_log, ensure_ascii=False, indent=2))
    temporary.replace(upload_log_path)
    if index % 50 == 0 or index == len(image_audit):
        print(f'이미지 처리 {index}/{len(image_audit)} · 업로드 {uploaded} · 재사용 {reused}', flush=True)
print('Storage 처리 완료')

## 9. 실제 DB 등록

**이 셀은 anna_rag의 5개 테이블에 데이터를 넣습니다.** 부모부터 등록합니다. 같은 키가 이미 있으면 건너뛰되, 먼저 기존 값이 동일한지 다시 확인합니다. 등록과 건수 검사가 모두 성공한 경우에만 커밋합니다. 임베딩 컬럼은 입력하지 않으므로 새 행의 값은 NULL입니다.

In [ ]:
with db.connect(camel_case_keys=False) as conn:
    conn.execute("SET LOCAL statement_timeout = '120s'")
    # 같은 문서의 동시 적재를 순서대로 실행하기 위한 DB 잠금입니다.
    conn.execute('SELECT pg_advisory_xact_lock(hashtext(%s))', (document_id,))
    check_existing(conn)
    for name, rows in tables.items():
        columns = list(rows[0])
        query = sql.SQL('INSERT INTO anna_rag.{} ({}) VALUES ({}) ON CONFLICT DO NOTHING').format(
            sql.Identifier(name), sql.SQL(',').join(map(sql.Identifier, columns)),
            sql.SQL(',').join(sql.Placeholder() for _ in columns))
        with conn.cursor() as cursor:
            cursor.executemany(query, [tuple(row[column] for column in columns) for row in rows])
        count_query = sql.SQL('SELECT count(*) AS count FROM anna_rag.{} WHERE document_id=%s').format(sql.Identifier(name))
        count = conn.execute(count_query, (document_id,)).fetchone()['count']
        assert count == len(rows), f'{name} 건수 불일치'
    check_existing(conn)
print('DB 등록 완료: 5개 테이블 전체 내용과 건수 일치')

## 10. DB에서 다시 읽어 부모와 그림 연결 확인

첫 청크를 임의로 선택하는 구조 확인입니다. 아직 임베딩 유사도 검색이 아닙니다. 실제 검색에서는 Top-K 청크의 순위를 유지하고, 부모는 `(document_id, section_id)`로 중복 제거한 뒤 별도의 토큰 예산 안에서 가져와야 합니다. 길이를 제한했다면 잘린 사실을 표시하고 필수 경고가 빠지지 않았는지 확인해야 합니다.

In [ ]:
with db.connect(camel_case_keys=False) as conn:
    conn.execute('SET TRANSACTION READ ONLY')
    counts = []
    for name in tables:
        query = sql.SQL('SELECT count(*) AS count FROM anna_rag.{} WHERE document_id=%s').format(sql.Identifier(name))
        counts.append({'테이블': name, 'DB 행 수': conn.execute(query, (document_id,)).fetchone()['count']})
    embedding_count = conn.execute('SELECT count(*) AS count FROM anna_rag.chunks WHERE document_id=%s AND embedding IS NOT NULL', (document_id,)).fetchone()['count']
    sample = conn.execute("""
        SELECT c.chunk_id, c.page_content AS child_text, s.title, s.full_text AS parent_text
        FROM anna_rag.chunks c JOIN anna_rag.sections s USING(document_id, section_id)
        WHERE c.document_id=%s AND c.section_id=%s AND c.chunk_type='text'
        ORDER BY c.chunk_order LIMIT 1
    """, (document_id, emergency_parent['section_id'])).fetchone()
    linked_images = conn.execute("""
        SELECT i.image_id, i.storage_path FROM anna_rag.section_images si
        JOIN anna_rag.images i USING(document_id, image_id)
        WHERE si.document_id=%s AND si.section_id=%s
    """, (document_id, emergency_parent['section_id'])).fetchall()
display(pd.DataFrame(counts))
print('임베딩이 있는 청크:', embedding_count)
print('조회한 부모:', sample['title'], '/ 전체 본문 길이:', len(sample['parent_text']))
print('연결 그림 수:', len(linked_images))
assert sample['parent_text'] == emergency_parent['full_text']
# 저장소의 대표 파일을 다시 받아 로컬 원본과 바이트가 같은지 검증합니다.
for image_id in [images[0]['image_id'], 'p018_image_01', images[-1]['image_id']]:
    info = image_lookup[image_id]
    downloaded = storage.supabase.storage.from_('images').download(info['storage_path'])
    local = image_dir / f'{image_id}.png'
    assert hashlib.sha256(downloaded).digest() == hashlib.sha256(local.read_bytes()).digest()
print('DB 부모 연결 및 Storage 대표 파일 재조회 검증 통과')
summary['database_counts'] = counts
summary['embedding_rows'] = embedding_count
summary['storage_sample_verified'] = True
(output_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2))
pdf.close()

## 11. Storage 전체 파일 목록 확인
대표 파일 바이트 검증에 더해 모든 예상 파일명이 서버에 있는지 확인합니다. 최초 적재 후 동일 INSERT를 한 번 더 실행하여 중복으로 행 수가 늘어나지 않는 것도 검증했습니다.


In [ ]:
# Storage 객체 목록을 페이지 단위로 읽어 모든 예상 파일이 존재하는지 확인합니다.
prefix = f'cars/hyundai/ioniq5/{document_id}'
remote_names = set()
offset = 0
while True:
    batch = storage.supabase.storage.from_('images').list(
        path=prefix, options={'limit': 100, 'offset': offset, 'sortBy': {'column': 'name', 'order': 'asc'}})
    remote_names.update(row['name'] for row in batch if row.get('id'))
    if len(batch) < 100:
        break
    offset += len(batch)
expected_names = {row['storage_path'].rsplit('/', 1)[-1] for row in tables['images']}
assert expected_names <= remote_names, 'Storage에 누락된 이미지가 있습니다.'
print('Storage 전체 예상 파일 확인:', len(expected_names), '/', len(expected_names))


## 다음 단계와 확인할 한계

- 현재 부모는 PDF의 공식 목차 경계입니다. 전체 상위 장을 무조건 합치지 않으며 부모 바깥 참조 문서는 자동 확장하지 않습니다.
- 전체 목차 위치와 본문 글자 커버리지는 자동 검증합니다. 표의 읽기 순서, 자동 이미지 캡션의 유용성은 실제 질문과 샘플 페이지로 추가 검토해야 합니다.
- 2pt 그림 식별자와 반복 페이지 하단은 검색 본문에서 제외했습니다. 별도 벡터 도형과 스캔 이미지 안의 글자는 OCR로 추출하지 않습니다.
- 다음에는 수업에서 배운 임베딩 도구를 연결하고 같은 Top-K에서 부모 문맥을 추가하는 검색 함수를 만듭니다. 평가 고객을 모집하는 작업은 포함하지 않습니다.